In [ ]:
# 1. Installs (run once)
!uv pip install -q sentence-transformers datasets faiss-cpu threadpoolctl accelerate scikit-learn


In [ ]:
# 2. Imports + setup
import random
from pathlib import Path

import faiss
import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import InputExample, SentenceTransformer, losses
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import get_linear_schedule_with_warmup

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using torch device: {device}")


In [ ]:
# 3. Load full MBPP dataset (all splits)
dataset = load_dataset("google-research-datasets/mbpp")
print({split: len(dataset[split]) for split in dataset.keys()})

split_names = ["train", "validation", "test", "prompt"]
all_pairs = []
for split_name in split_names:
    for row in dataset[split_name]:
        query = row["text"].strip()
        code = row["code"].strip()
        if query and code:
            all_pairs.append(
                {
                    "query": query,
                    "code": code,
                    "task_id": row["task_id"],
                    "split": split_name,
                }
            )

# Stable ordering for reproducible retrieval labels.
all_pairs = sorted(all_pairs, key=lambda x: (x["task_id"], x["split"]))

query_texts = [p["query"] for p in all_pairs]
corpus_codes = [p["code"] for p in all_pairs]
num_docs = len(all_pairs)

print(f"Total usable query-code pairs: {num_docs}")


In [ ]:
# 4. Retrieval metrics + baseline evaluators
def compute_retrieval_metrics_from_ranks(
    ranks,
    num_docs,
    k_values=(1, 5, 10, 20),
    include_f1=False,
):
    """
    ranks: 1-based rank of the true code for each query.

    We keep F1 optional because with exactly one relevant doc/query,
    F1@k becomes mostly a monotonic transform of hit@k and k.
    """
    ranks = np.asarray(ranks, dtype=np.int32)
    metrics = {
        "num_queries": int(len(ranks)),
        "num_docs": int(num_docs),
        "mrr": float(np.mean(1.0 / ranks)),
    }

    for k in k_values:
        hits = (ranks <= k).astype(np.float32)
        recall_at_k = float(np.mean(hits))
        precision_at_k = float(np.mean(hits / k))

        ap_at_k = np.where(ranks <= k, 1.0 / ranks, 0.0)
        map_at_k = float(np.mean(ap_at_k))

        ndcg_at_k = np.where(ranks <= k, 1.0 / np.log2(ranks + 1), 0.0)
        ndcg_at_k = float(np.mean(ndcg_at_k))

        metrics[f"recall@{k}"] = recall_at_k
        metrics[f"precision@{k}"] = precision_at_k
        metrics[f"map@{k}"] = map_at_k
        metrics[f"ndcg@{k}"] = ndcg_at_k

        if include_f1:
            f1_vals = np.where(
                hits > 0,
                2.0 * (1.0 / k) * 1.0 / ((1.0 / k) + 1.0),
                0.0,
            )
            metrics[f"f1@{k}"] = float(np.mean(f1_vals))

    return metrics


def evaluate_dense_embeddings(query_embeddings, code_embeddings, k_values=(1, 5, 10, 20), include_f1=False):
    query_embeddings = np.asarray(query_embeddings, dtype="float32")
    code_embeddings = np.asarray(code_embeddings, dtype="float32")

    index = faiss.IndexFlatIP(code_embeddings.shape[1])
    index.add(code_embeddings)

    # Retrieve full ranking so MRR/MAP/nDCG are exact.
    _, neighbors = index.search(query_embeddings, code_embeddings.shape[0])

    ranks = np.empty(query_embeddings.shape[0], dtype=np.int32)
    for i, retrieved in enumerate(neighbors):
        ranks[i] = int(np.where(retrieved == i)[0][0]) + 1

    metrics = compute_retrieval_metrics_from_ranks(
        ranks,
        num_docs=code_embeddings.shape[0],
        k_values=k_values,
        include_f1=include_f1,
    )
    metrics["mean_rank"] = float(np.mean(ranks))
    return metrics


def evaluate_tfidf_baseline(queries, corpus, k_values=(1, 5, 10, 20), include_f1=False):
    vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), lowercase=True)
    code_tfidf = vectorizer.fit_transform(corpus)
    query_tfidf = vectorizer.transform(queries)

    sim = (query_tfidf @ code_tfidf.T).toarray()
    order = np.argsort(-sim, axis=1)

    ranks = np.empty(sim.shape[0], dtype=np.int32)
    for i in range(sim.shape[0]):
        ranks[i] = int(np.where(order[i] == i)[0][0]) + 1

    metrics = compute_retrieval_metrics_from_ranks(
        ranks,
        num_docs=len(corpus),
        k_values=k_values,
        include_f1=include_f1,
    )
    metrics["mean_rank"] = float(np.mean(ranks))
    return metrics


def evaluate_random_baseline(num_queries, num_docs, k_values=(1, 5, 10, 20), include_f1=False):
    # Exact random baseline for one relevant item among num_docs candidates.
    possible_ranks = np.arange(1, num_docs + 1, dtype=np.float32)
    probs = np.ones(num_docs, dtype=np.float32) / num_docs

    metrics = {
        "num_queries": int(num_queries),
        "num_docs": int(num_docs),
        "mrr": float(np.sum((1.0 / possible_ranks) * probs)),
    }

    for k in k_values:
        hits = (possible_ranks <= k).astype(np.float32)
        metrics[f"recall@{k}"] = float(np.sum(hits * probs))
        metrics[f"precision@{k}"] = float(np.sum((hits / k) * probs))
        metrics[f"map@{k}"] = float(np.sum(np.where(possible_ranks <= k, 1.0 / possible_ranks, 0.0) * probs))
        metrics[f"ndcg@{k}"] = float(
            np.sum(np.where(possible_ranks <= k, 1.0 / np.log2(possible_ranks + 1), 0.0) * probs)
        )

        if include_f1:
            f1_vals = np.where(hits > 0, 2.0 * (1.0 / k) / ((1.0 / k) + 1.0), 0.0)
            metrics[f"f1@{k}"] = float(np.sum(f1_vals * probs))

    metrics["mean_rank"] = float(np.sum(possible_ranks * probs))
    return metrics


def format_texts_for_model(model_name, queries, codes):
    name = model_name.lower()
    if "e5" in name:
        return [f"query: {q}" for q in queries], [f"passage: {c}" for c in codes]
    if "bge" in name:
        instruction = "Represent this sentence for searching relevant code snippets: "
        return [instruction + q for q in queries], codes
    return queries, codes


def metric_snapshot(metrics, ks=(1, 5, 10)):
    out = {"mrr": metrics["mrr"]}
    for k in ks:
        out[f"recall@{k}"] = metrics[f"recall@{k}"]
        out[f"map@{k}"] = metrics[f"map@{k}"]
        out[f"ndcg@{k}"] = metrics[f"ndcg@{k}"]
    return out


In [ ]:
# 5. Baseline metrics (random + TF-IDF lexical)
k_values = (1, 5, 10, 20)
include_f1_metrics = False  # Kept off by default; see note in metric function.

random_metrics = evaluate_random_baseline(
    num_queries=num_docs,
    num_docs=num_docs,
    k_values=k_values,
    include_f1=include_f1_metrics,
)

tfidf_metrics = evaluate_tfidf_baseline(
    query_texts,
    corpus_codes,
    k_values=k_values,
    include_f1=include_f1_metrics,
)

baseline_results = [
    {"model": "random", **metric_snapshot(random_metrics)},
    {"model": "tfidf_lexical", **metric_snapshot(tfidf_metrics)},
]

print("Baseline results:")
for row in baseline_results:
    print(row)


In [ ]:
# 6. Zero-shot dense embedding model benchmarks (full dataset)
model_candidates = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "sentence-transformers/multi-qa-mpnet-base-dot-v1",
    "BAAI/bge-base-en-v1.5",
    "intfloat/e5-base-v2",
]

run_model_benchmarks = True
model_benchmark_results = []

if run_model_benchmarks:
    for model_name in model_candidates:
        print(f"\nEvaluating {model_name} ...")
        model = SentenceTransformer(model_name)
        model.to(device)

        q_texts, c_texts = format_texts_for_model(model_name, query_texts, corpus_codes)
        q_emb = model.encode(
            q_texts,
            batch_size=64,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        c_emb = model.encode(
            c_texts,
            batch_size=64,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True,
        )

        metrics = evaluate_dense_embeddings(
            q_emb,
            c_emb,
            k_values=k_values,
            include_f1=include_f1_metrics,
        )

        row = {"model": model_name, **metric_snapshot(metrics)}
        model_benchmark_results.append(row)
        print(row)

    model_benchmark_results = sorted(model_benchmark_results, key=lambda x: x["mrr"], reverse=True)
    print("\nDense model ranking by MRR:")
    for row in model_benchmark_results:
        print(row)

selected_model_name = (
    model_benchmark_results[0]["model"]
    if model_benchmark_results
    else "sentence-transformers/all-MiniLM-L6-v2"
)
print(f"\nSelected model for fine-tuning: {selected_model_name}")


In [ ]:
# 7. Full-dataset training data + manual MNR training function
train_examples = [InputExample(texts=[p["query"], p["code"]]) for p in all_pairs]
print(f"Training examples (full dataset): {len(train_examples)}")


def finetune_with_mnr(
    model_name,
    train_examples,
    epochs=1,
    batch_size=16,
    lr=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
):
    model = SentenceTransformer(model_name)
    model.to(device)

    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size, drop_last=True)
    train_dataloader.collate_fn = model.smart_batching_collate

    train_loss = losses.MultipleNegativesRankingLoss(model=model)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = epochs * len(train_dataloader)
    warmup_steps = max(1, int(total_steps * warmup_ratio))
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    model.train()
    epoch_losses = []

    for epoch in range(epochs):
        running_loss = 0.0
        progress = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{epochs}")
        for features, labels in progress:
            optimizer.zero_grad()
            loss_value = train_loss(features, labels)
            loss_value.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            scheduler.step()

            running_loss += float(loss_value.item())
            progress.set_postfix(loss=f"{loss_value.item():.4f}")

        avg_epoch_loss = running_loss / max(1, len(train_dataloader))
        epoch_losses.append(avg_epoch_loss)

    training_stats = {
        "total_steps": int(total_steps),
        "warmup_steps": int(warmup_steps),
        "epoch_losses": epoch_losses,
        "mean_epoch_loss": float(np.mean(epoch_losses)),
    }
    return model, training_stats


In [ ]:
# 8. Hyperparameter exploration (optional)
sweep_configs = [
    {"epochs": 1, "batch_size": 16, "lr": 2e-5, "warmup_ratio": 0.10, "weight_decay": 0.01},
    {"epochs": 2, "batch_size": 16, "lr": 2e-5, "warmup_ratio": 0.10, "weight_decay": 0.01},
    {"epochs": 1, "batch_size": 32, "lr": 1e-5, "warmup_ratio": 0.06, "weight_decay": 0.01},
]

run_hyperparameter_sweep = False
sweep_results = []

if run_hyperparameter_sweep:
    for config in sweep_configs:
        print(f"\nRunning config: {config}")
        model_ft, train_stats = finetune_with_mnr(
            selected_model_name,
            train_examples,
            epochs=config["epochs"],
            batch_size=config["batch_size"],
            lr=config["lr"],
            warmup_ratio=config["warmup_ratio"],
            weight_decay=config["weight_decay"],
        )

        q_texts, c_texts = format_texts_for_model(selected_model_name, query_texts, corpus_codes)
        q_emb = model_ft.encode(q_texts, batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
        c_emb = model_ft.encode(c_texts, batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)
        metrics = evaluate_dense_embeddings(q_emb, c_emb, k_values=k_values, include_f1=include_f1_metrics)

        result = {
            "config": config,
            "train_stats": train_stats,
            "metrics": metrics,
            "summary": metric_snapshot(metrics),
        }
        sweep_results.append(result)
        print(result["summary"])

    sweep_results = sorted(sweep_results, key=lambda x: x["metrics"]["mrr"], reverse=True)

best_config = sweep_results[0]["config"] if sweep_results else sweep_configs[0]
print(f"Best config for final training: {best_config}")


In [ ]:
# 9. Final full-dataset fine-tuning + evaluation
run_final_training = False

final_model = None
final_train_stats = None
final_metrics = None

if not run_final_training:
    print("Final training is disabled by default. Set run_final_training=True to train on the full dataset.")
model_output_dir = "artifacts/mbpp_embedder_full_dataset"

if run_final_training:
    final_model, final_train_stats = finetune_with_mnr(
        selected_model_name,
        train_examples,
        epochs=best_config["epochs"],
        batch_size=best_config["batch_size"],
        lr=best_config["lr"],
        warmup_ratio=best_config["warmup_ratio"],
        weight_decay=best_config["weight_decay"],
    )

    Path(model_output_dir).mkdir(parents=True, exist_ok=True)
    final_model.save(model_output_dir)

    q_texts, c_texts = format_texts_for_model(selected_model_name, query_texts, corpus_codes)
    corpus_embeddings = final_model.encode(
        c_texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    query_embeddings = final_model.encode(
        q_texts,
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    final_metrics = evaluate_dense_embeddings(
        query_embeddings,
        corpus_embeddings,
        k_values=k_values,
        include_f1=include_f1_metrics,
    )

    index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
    index.add(corpus_embeddings.astype("float32"))

    mrr = final_metrics["mrr"]
    recall_at_1 = final_metrics["recall@1"]
    recall_at_5 = final_metrics["recall@5"]
    recall_at_10 = final_metrics["recall@10"]

    print("Final model summary:")
    print(metric_snapshot(final_metrics))
    print(f"Saved model to: {model_output_dir}")


In [ ]:
# 10. Consolidated report + what affects metrics + sanity retrieval
all_results = []
all_results.extend(baseline_results)
all_results.extend(model_benchmark_results)
if final_metrics is not None:
    all_results.append({"model": f"finetuned::{selected_model_name}", **metric_snapshot(final_metrics)})

all_results = sorted(all_results, key=lambda x: x["mrr"], reverse=True)

print("=== Retrieval Results (sorted by MRR) ===")
for row in all_results:
    print(row)

print("\n=== Factors That Strongly Affect Retrieval Metrics ===")
factors = [
    "Embedding model pretraining objective/domain match (generic vs retrieval-focused vs instruction-tuned)",
    "Query/document text formatting (e.g., e5 query:/passage: prefixes, instruction prompts)",
    "Similarity function and normalization (cosine with L2 norm vs raw dot product)",
    "Batch size for contrastive loss (larger batches create more in-batch negatives)",
    "Learning rate, warmup ratio, epochs, weight decay, and gradient clipping",
    "Code/query preprocessing choices (whitespace normalization, truncation, comments/docstrings)",
    "Hard-negative strategy (currently absent; adding hard negatives usually improves discriminative ranking)",
]
for i, factor in enumerate(factors, start=1):
    print(f"{i}. {factor}")

if final_model is None:
    print("\nNo fine-tuned model run yet. Enable run_final_training=True in Cell 9 to train/evaluate on the full dataset.")

if final_model is not None:
    query = "reverse a string"
    q_formatted, _ = format_texts_for_model(selected_model_name, [query], corpus_codes[:1])
    q_emb = final_model.encode(q_formatted, convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_emb, 5)

    print(f"\nSanity query: {query}")
    for rank, (doc_id, score) in enumerate(zip(ids[0], scores[0]), start=1):
        snippet = corpus_codes[doc_id].replace("\n", " ").strip()
        if len(snippet) > 160:
            snippet = snippet[:160] + "..."
        print(f"{rank}. score={score:.4f} | task_id={all_pairs[doc_id]['task_id']} | {snippet}")
